### Project 1 — Metadata Experiments
### Objective
- Determine whether legitimate intake-time metadata provides predictive
- information beyond the complaint narrative for CFPB Product classification.
### Fixed NLP backbone:- Word TF-IDF (1,2) + Character TF-IDF (3,5)
### Target :- Product
### Primary metric:- Macro F1
### Experimental rule:- The test set remains untouched.
### Experiments
- 0. Narrative only — current benchmark
- 1. Narrative + Company
- 2. Narrative + Company + State
- 3. Narrative + Company + State + Date features
- 4. Narrative + Company + State + Date + ZIP
- 5. Narrative + Company + State + Date + Tags

### Imports and Configuration

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# project paths
PROJECT_ROOT = Path.cwd().parent

TRAIN_PATH = PROJECT_ROOT / "Data" / "processed" / "project1_train.csv"
VAL_PATH = PROJECT_ROOT / "Data" / "processed" / "project1_validation.csv"
TEST_PATH = PROJECT_ROOT / "Data" / "processed" / "project1_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (58422, 18)
Validation: (11685, 18)
Test: (11685, 18)


### Basic Validation

In [2]:
TARGET = "Product"
TEXT_COL = "Consumer complaint narrative"

assert TARGET in train_df.columns
assert TARGET in val_df.columns
assert TARGET in test_df.columns

assert TEXT_COL in train_df.columns
assert TEXT_COL in val_df.columns
assert TEXT_COL in test_df.columns

assert train_df[TARGET].notna().all()
assert val_df[TARGET].notna().all()
assert test_df[TARGET].notna().all()

print("Basic validation: PASS")
print("Product classes:", train_df[TARGET].nunique())

Basic validation: PASS
Product classes: 11


### Prepare Target and Narrative

In [3]:
y_train = train_df[TARGET]
y_val = val_df[TARGET]

X_train_text = train_df[TEXT_COL].fillna("").astype(str)
X_val_text = val_df[TEXT_COL].fillna("").astype(str)

print("Training rows:", len(y_train))
print("Validation rows:", len(y_val))
print("Classes:", y_train.nunique())

Training rows: 58422
Validation rows: 11685
Classes: 11


### Fixed NLP Representation

In [4]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_features=10000
)

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3,
    sublinear_tf=True,
    max_features=10000
)

text_vectorizer = FeatureUnion([
    ("word", word_vectorizer),
    ("char", char_vectorizer)
])

X_train_text_tfidf = text_vectorizer.fit_transform(X_train_text)
X_val_text_tfidf = text_vectorizer.transform(X_val_text)

print("Train matrix:", X_train_text_tfidf.shape)
print("Validation matrix:", X_val_text_tfidf.shape)

Train matrix: (58422, 20000)
Validation matrix: (11685, 20000)


### Evaluation Function

In [5]:
def evaluate_model(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            average="macro"
        ),
        "Weighted F1": f1_score(
            y_true,
            y_pred,
            average="weighted"
        )
    }

### Experiment 0: Current Benchmark

In [6]:
baseline_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

baseline_model.fit(
    X_train_text_tfidf,
    y_train
)

baseline_pred = baseline_model.predict(
    X_val_text_tfidf
)

results = []

results.append(
    evaluate_model(
        "0. Narrative only",
        y_val,
        baseline_pred
    )
)

pd.DataFrame(results)

,Model,Accuracy,Macro F1,Weighted F1
0,0. Narrative only,0.839196,0.745642,0.834671


### Metadata Eligibility Audit

In [7]:
metadata_cols = [
    "Company",
    "State",
    "ZIP code",
    "Tags",
    "Date received"
]

metadata_audit = []

for col in metadata_cols:
    metadata_audit.append({
        "Feature": col,
        "Train Missing %": train_df[col].isna().mean() * 100,
        "Validation Missing %": val_df[col].isna().mean() * 100,
        "Train Unique": train_df[col].nunique(dropna=True),
        "Validation Unique": val_df[col].nunique(dropna=True)
    })

metadata_audit_df = pd.DataFrame(metadata_audit)

metadata_audit_df

,Feature,Train Missing %,Validation Missing %,Train Unique,Validation Unique
0,Company,0.000000,0.000000,1751,889
1,State,0.730889,0.624733,57,56
2,ZIP code,0.421074,0.359435,6628,4591
3,Tags,84.594844,84.107831,3,3
4,Date received,0.000000,0.000000,57797,11626


### Date Feature Engineering

In [8]:
def create_date_features(df):
    dates = pd.to_datetime(
        df["Date received"],
        errors="coerce"
    )

    features = pd.DataFrame(index=df.index)

    features["received_month"] = dates.dt.month
    features["received_dayofweek"] = dates.dt.dayofweek
    features["received_day"] = dates.dt.day

    return features

### Create Date Features

In [9]:
train_date = create_date_features(train_df)
val_date = create_date_features(val_df)

print(train_date.head())

   received_month  received_dayofweek  received_day
0               4                   2            29
1               4                   3            30
2               4                   2            29
3               6                   0            15
4               6                   0            15


### Categorical Encoder

In [10]:
categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

### Helper Function for Metadata

In [11]:
def prepare_categorical_metadata(
    train_df,
    val_df,
    columns
):
    train_meta = train_df[columns].copy()
    val_meta = val_df[columns].copy()

    for col in columns:
        train_meta[col] = train_meta[col].fillna("__MISSING__").astype(str)
        val_meta[col] = val_meta[col].fillna("__MISSING__").astype(str)

    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )

    train_encoded = encoder.fit_transform(train_meta)
    val_encoded = encoder.transform(val_meta)

    return train_encoded, val_encoded, encoder

### Experiment 1: Narrative + Company
- Features: 
 - Complaint narrative 
 - Company
##### Question:-> Does company identity provide predictive information beyond the narrative?

In [12]:
X_train_company, X_val_company, company_encoder = (
    prepare_categorical_metadata(
        train_df,
        val_df,
        ["Company"]
    )
)

X_train_exp1 = FeatureUnion([
    ("text", text_vectorizer)
]).transform(X_train_text)

X_val_exp1 = FeatureUnion([
    ("text", text_vectorizer)
]).transform(X_val_text)

In [13]:
from scipy.sparse import hstack

X_train_exp1 = hstack([
    X_train_text_tfidf,
    X_train_company
])

X_val_exp1 = hstack([
    X_val_text_tfidf,
    X_val_company
])

model_exp1 = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model_exp1.fit(
    X_train_exp1,
    y_train
)

pred_exp1 = model_exp1.predict(
    X_val_exp1
)

results.append(
    evaluate_model(
        "1. Narrative + Company",
        y_val,
        pred_exp1
    )
)

pd.DataFrame(results)

,Model,Accuracy,Macro F1,Weighted F1
0,0. Narrative only,0.839196,0.745642,0.834671
1,1. Narrative + Company,0.868036,0.804323,0.866081


#### Experiment 2: + state
- Features:
- Complaint narrative
- Company
- State
- Question:-> Does geographic information add predictive signal beyond narrative + company?

In [14]:
X_train_company_state, X_val_company_state, _ = (
    prepare_categorical_metadata(
        train_df,
        val_df,
        ["Company", "State"]
    )
)

X_train_exp2 = hstack([
    X_train_text_tfidf,
    X_train_company_state
])

X_val_exp2 = hstack([
    X_val_text_tfidf,
    X_val_company_state
])

model_exp2 = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model_exp2.fit(
    X_train_exp2,
    y_train
)

pred_exp2 = model_exp2.predict(
    X_val_exp2
)

results.append(
    evaluate_model(
        "2. Narrative + Company + State",
        y_val,
        pred_exp2
    )
)

pd.DataFrame(results)

,Model,Accuracy,Macro F1,Weighted F1
0,0. Narrative only,0.839196,0.745642,0.834671
1,1. Narrative + Company,0.868036,0.804323,0.866081
2,2. Narrative + Company + State,0.868977,0.803920,0.866984


## Experiment 3: + Date Features
- Features:
- Complaint narrative
- Company
- State
- Date received features
- Question:-> Does intake-time temporal context provide additional predictive signal?

In [15]:
date_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

train_date_encoded = date_encoder.fit_transform(
    train_date.astype(str)
)

val_date_encoded = date_encoder.transform(
    val_date.astype(str)
)

X_train_exp3 = hstack([
    X_train_text_tfidf,
    X_train_company_state,
    train_date_encoded
])

X_val_exp3 = hstack([
    X_val_text_tfidf,
    X_val_company_state,
    val_date_encoded
])

model_exp3 = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model_exp3.fit(
    X_train_exp3,
    y_train
)

pred_exp3 = model_exp3.predict(
    X_val_exp3
)

results.append(
    evaluate_model(
        "3. Narrative + Company + State + Date",
        y_val,
        pred_exp3
    )
)

pd.DataFrame(results)

,Model,Accuracy,Macro F1,Weighted F1
0,0. Narrative only,0.839196,0.745642,0.834671
1,1. Narrative + Company,0.868036,0.804323,0.866081
2,2. Narrative + Company + State,0.868977,0.803920,0.866984
3,3. Narrative + Company + State + Date,0.868806,0.801952,0.866695


## Experiment 4: ZIP
- Features:
- Narrative
- Company
- State
- Date
- ZIP
- Question:-> Does ZIP provide additional predictive signal?
- Caution:- ZIP is high-cardinality and may act as a geographic shortcut.

In [16]:
zip_train = train_df[["ZIP code"]].copy()
zip_val = val_df[["ZIP code"]].copy()

zip_train["ZIP code"] = (
    zip_train["ZIP code"]
    .fillna("__MISSING__")
    .astype(str)
)

zip_val["ZIP code"] = (
    zip_val["ZIP code"]
    .fillna("__MISSING__")
    .astype(str)
)

zip_encoder = OneHotEncoder(
    handle_unknown="ignore",
    min_frequency=5,
    sparse_output=True
)

train_zip_encoded = zip_encoder.fit_transform(zip_train)
val_zip_encoded = zip_encoder.transform(zip_val)

### Now The Model

In [17]:
X_train_exp4 = hstack([
    X_train_text_tfidf,
    X_train_company_state,
    train_date_encoded,
    train_zip_encoded
])

X_val_exp4 = hstack([
    X_val_text_tfidf,
    X_val_company_state,
    val_date_encoded,
    val_zip_encoded
])

model_exp4 = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model_exp4.fit(
    X_train_exp4,
    y_train
)

pred_exp4 = model_exp4.predict(
    X_val_exp4
)

results.append(
    evaluate_model(
        "4. Narrative + Company + State + Date + ZIP",
        y_val,
        pred_exp4
    )
)

pd.DataFrame(results)

,Model,Accuracy,Macro F1,Weighted F1
0,0. Narrative only,0.839196,0.745642,0.834671
1,1. Narrative + Company,0.868036,0.804323,0.866081
2,2. Narrative + Company + State,0.868977,0.803920,0.866984
3,3. Narrative + Company + State + Date,0.868806,0.801952,0.866695
4,4. Narrative + Company + State + Date + ZIP,0.866838,0.798262,0.864569


#### Experiment 5: Tags
- Features:
- Narrative
- Company
- State
- Date
- Tags
- Question:-> Do CFPB tags provide useful predictive information at intake?
- Important:- Tags are only legitimate if their assignment is available at T_pred.

In [18]:
tags_train = train_df[["Tags"]].copy()
tags_val = val_df[["Tags"]].copy()

tags_train["Tags"] = (
    tags_train["Tags"]
    .fillna("__MISSING__")
    .astype(str)
)

tags_val["Tags"] = (
    tags_val["Tags"]
    .fillna("__MISSING__")
    .astype(str)
)

tags_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

train_tags_encoded = tags_encoder.fit_transform(tags_train)
val_tags_encoded = tags_encoder.transform(tags_val)

### Model

In [19]:
X_train_exp5 = hstack([
    X_train_text_tfidf,
    X_train_company_state,
    train_date_encoded,
    train_tags_encoded
])

X_val_exp5 = hstack([
    X_val_text_tfidf,
    X_val_company_state,
    val_date_encoded,
    val_tags_encoded
])

model_exp5 = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

model_exp5.fit(
    X_train_exp5,
    y_train
)

pred_exp5 = model_exp5.predict(
    X_val_exp5
)

results.append(
    evaluate_model(
        "5. Narrative + Company + State + Date + Tags",
        y_val,
        pred_exp5
    )
)

pd.DataFrame(results)

,Model,Accuracy,Macro F1,Weighted F1
0,0. Narrative only,0.839196,0.745642,0.834671
1,1. Narrative + Company,0.868036,0.804323,0.866081
2,2. Narrative + Company + State,0.868977,0.803920,0.866984
3,3. Narrative + Company + State + Date,0.868806,0.801952,0.866695
4,4. Narrative + Company + State + Date + ZIP,0.866838,0.798262,0.864569
5,5. Narrative + Company + State + Date + Tags,0.868122,0.801726,0.866063


### Final Experiment Comparison

In [20]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Accuracy,Macro F1,Weighted F1
0,1. Narrative + Company,0.868036,0.804323,0.866081
1,2. Narrative + Company + State,0.868977,0.803920,0.866984
2,3. Narrative + Company + State + Date,0.868806,0.801952,0.866695
3,5. Narrative + Company + State + Date + Tags,0.868122,0.801726,0.866063
4,4. Narrative + Company + State + Date + ZIP,0.866838,0.798262,0.864569
5,0. Narrative only,0.839196,0.745642,0.834671


### Improvement Over Current Benchmark

In [21]:
benchmark_macro_f1 = 0.7461

results_df["Macro F1 Improvement"] = (
    results_df["Macro F1"] - benchmark_macro_f1
)

results_df["Macro F1 Improvement %"] = (
    results_df["Macro F1 Improvement"]
    / benchmark_macro_f1
    * 100
)

results_df

,Model,Accuracy,Macro F1,Weighted F1,Macro F1 Improvement,Macro F1 Improvement %
0,1. Narrative + Company,0.868036,0.804323,0.866081,0.058223,7.803591
1,2. Narrative + Company + State,0.868977,0.803920,0.866984,0.057820,7.749667
2,3. Narrative + Company + State + Date,0.868806,0.801952,0.866695,0.055852,7.485794
3,5. Narrative + Company + State + Date + Tags,0.868122,0.801726,0.866063,0.055626,7.455560
4,4. Narrative + Company + State + Date + ZIP,0.866838,0.798262,0.864569,0.052162,6.991278
5,0. Narrative only,0.839196,0.745642,0.834671,-0.000458,-0.061410


### Identify Best Experiment

In [22]:
best_row = results_df.iloc[0]

print("Best metadata experiment:")
print(best_row["Model"])

print(f"\nAccuracy: {best_row['Accuracy']:.4f}")
print(f"Macro F1: {best_row['Macro F1']:.4f}")
print(f"Weighted F1: {best_row['Weighted F1']:.4f}")

print(
    f"Macro F1 improvement: "
    f"{best_row['Macro F1 Improvement']:.4f}"
)

Best metadata experiment:
1. Narrative + Company

Accuracy: 0.8680
Macro F1: 0.8043
Weighted F1: 0.8661
Macro F1 improvement: 0.0582


### Per-Class Report for Best Experiment

In [23]:
prediction_map = {
    "0. Narrative only": baseline_pred,
    "1. Narrative + Company": pred_exp1,
    "2. Narrative + Company + State": pred_exp2,
    "3. Narrative + Company + State + Date": pred_exp3,
    "4. Narrative + Company + State + Date + ZIP": pred_exp4,
    "5. Narrative + Company + State + Date + Tags": pred_exp5
}

best_predictions = prediction_map[best_row["Model"]]

print(
    classification_report(
        y_val,
        best_predictions,
        digits=4
    )
)

                                                         precision    recall  f1-score   support

                            Checking or savings account     0.7968    0.8683    0.8311      2096
                                            Credit card     0.8552    0.8665    0.8608      1970
    Credit reporting or other personal consumer reports     0.8274    0.8742    0.8502       477
                                        Debt collection     0.9270    0.9366    0.9317      3848
                              Debt or credit management     0.9250    0.3592    0.5175       103
     Money transfer, virtual currency, or money service     0.7707    0.6973    0.7322       935
                                               Mortgage     0.9524    0.9449    0.9486       762
Payday loan, title loan, personal loan, or advance loan     0.7486    0.6578    0.7003       412
                                           Prepaid card     0.8090    0.5538    0.6575       130
                             

# Project 1 — Metadata Experiment Decision Gate

## Questions

1. Did metadata improve Macro F1?
2. Was the improvement meaningful?
3. Which metadata contributed the improvement?
4. Did minority-class recall improve?
5. Did any metadata create suspicious shortcut behavior?
6. Are Company/State/ZIP/Tags genuinely available at T_pred?
7. Does the improvement justify additional model complexity?

### Decision rule

If metadata provides only negligible improvement:

→ Keep the simpler narrative-first model.

If metadata provides meaningful improvement without leakage:

→ Freeze the improved feature set.

If one feature produces a suspiciously large jump:

→ Investigate that feature before accepting it.

The final test set remains untouched.

### Automated Decision Summary

In [24]:
best_macro_f1 = results_df.iloc[0]["Macro F1"]

if best_macro_f1 > benchmark_macro_f1 + 0.005:
    decision = (
        "Metadata provides a potentially meaningful improvement. "
        "Investigate the winning feature set before freezing."
    )
else:
    decision = (
        "Metadata improvement is small. "
        "Prefer the simpler narrative-first model unless business "
        "value justifies the added complexity."
    )

print("Decision:")
print(decision)

Decision:
Metadata provides a potentially meaningful improvement. Investigate the winning feature set before freezing.


### Analysis Insights
### Feature Representation
- The combined Word + Character TF-IDF representation remains the fixed NLP backbone because it outperformed the individual representations.
### Metadata
- Metadata was introduced incrementally rather than all at once to identify
the marginal contribution of each feature group.
### Leakage Control
- Only information legitimately available at the prediction point was considered. Downstream taxonomy and outcome fields remain excluded.
### Generalization
- The validation set was used as the experimental decision environment.The final test set remains untouched.
### Model Selection
- The winning feature set will be frozen before hyperparameter tuning.
### Next Stage
- After metadata evaluation:
- 1. Freeze feature set
- 2. Hyperparameter tuning
- 3. Error analysis
- 4. Final model selection
- 5. Evaluate once on untouched test set